# 03. Sistema de Recomendación Basado en Metadatos Enriquecidos (Metadata Soup)

## Introducción
Para superar las limitaciones del recomendador anterior, incorporaremos metadatos más descriptivos y de mayor calidad:
* **Los 3 actores principales** (`cast`).
* **El director** (`crew`).
* **Palabras clave de la trama** (`keywords`).
* **Géneros cinematográficos** (`genres`).

A partir de estas variables construiremos una *"sopa de metadatos"* (*metadata soup*), que reúne en una sola cadena de texto los atributos clave de cada película.
Luego, aplicaremos `CountVectorizer` y `cosine_similarity` para generar recomendaciones mucho más precisas y coherentes con la autoría y el estilo cinematográfico.

In [3]:
import pandas as pd
import numpy as np
import os
from ast import literal_eval
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Carga de Datos y Verificación
Cargaremos los tres archivos necesarios: `movies_metadata.csv`, `credits.csv` y `keywords.csv`.

In [4]:
# Carga del catálogo principal y palabras clave
df = pd.read_csv('./data/movies_metadata.csv', low_memory=False)
keywords = pd.read_csv('./data/keywords.csv')

# Verificar disponibilidad de credits.csv
if not os.path.exists('./data/credits.csv'):
    raise FileNotFoundError(
        "El archivo './data/credits.csv' no fue encontrado en la carpeta 'data/'. "
        "Descárgalo del dataset de MovieLens y colócalo en './data/credits.csv'."
    )
else:
    credits = pd.read_csv('./data/credits.csv')
    print("Archivos cargados correctamente.")

Archivos cargados correctamente.


## 2. Limpieza de IDs y Fusión de Datasets (Merge)
Existen algunos registros con formato inconsistente en los IDs de `df` que debemos remover antes de convertir los IDs a enteros.

In [5]:
# Eliminar IDs problemáticos del DataFrame principal
df = df.drop([19730, 29503, 35587, 35803], errors='ignore')

# Convertir la columna 'id' a entero en los tres dataframes
keywords['id'] = keywords['id'].astype('int')
credits['id'] = credits['id'].astype('int')
df['id'] = df['id'].astype('int')

# Combinar dataframes mediante 'id'
df = df.merge(credits, on='id')
df = df.merge(keywords, on='id')

print("Dimensiones después del merge:", df.shape)
df[['title', 'cast', 'crew', 'keywords', 'genres']].head(2)

Dimensiones después del merge: (46626, 27)


,title,cast,crew,keywords,genres
0,Toy Story,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...","[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '..."
1,Jumanji,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...","[{'id': 10090, 'name': 'board game'}, {'id': 1...","[{'id': 12, 'name': 'Adventure'}, {'id': 14, '..."


## 3. Parseo de Cadenas Serializadas a Objetos Nativos de Python
Las columnas `cast`, `crew`, `keywords` y `genres` están guardadas como cadenas de texto que representan listas y diccionarios JSON.
Utilizamos `ast.literal_eval` para convertirlas de forma segura en listas de diccionarios de Python.

In [6]:
# Parsear las columnas serializadas a objetos Python
features = ['cast', 'crew', 'keywords', 'genres']
for feature in features:
    df[feature] = df[feature].apply(literal_eval)

## 4. Extracción de Metadatos Relevantes
Definimos funciones auxiliares para:
1. `get_director`: Extraer el nombre del director buscando `'job' == 'Director'` en la lista del equipo (`crew`). Si no está, asigna `np.nan`.
2. `get_list`: Extraer los primeros 3 elementos de una lista (`cast`, `keywords`, `genres`).

In [7]:
# Función para obtener el director
def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan

# Función para extraer hasta los 3 primeros nombres
def get_list(x):
    if isinstance(x, list):
        names = [i['name'] for i in x]
        if len(names) > 3:
            names = names[:3]
        return names
    return []

# Aplicar las funciones al DataFrame
df['director'] = df['crew'].apply(get_director)

features = ['cast', 'keywords', 'genres']
for feature in features:
    df[feature] = df[feature].apply(get_list)

# Visualizar las columnas resultantes
df[['title', 'cast', 'director', 'keywords', 'genres']].head(3)

,title,cast,director,keywords,genres
0,Toy Story,"[Tom Hanks, Tim Allen, Don Rickles]",John Lasseter,"[jealousy, toy, boy]","[Animation, Comedy, Family]"
1,Jumanji,"[Robin Williams, Jonathan Hyde, Kirsten Dunst]",Joe Johnston,"[board game, disappearance, based on children'...","[Adventure, Fantasy, Family]"
2,Grumpier Old Men,"[Walter Matthau, Jack Lemmon, Ann-Margret]",Howard Deutch,"[fishing, best friend, duringcreditsstinger]","[Romance, Comedy]"


## 5. Limpieza y Normalización de Texto
Para evitar que el vectorizador confunda personas con nombres o apellidos compartidos (por ejemplo, *Johnny Depp* vs *Johnny Galecki*), eliminamos los espacios y convertimos todo a minúsculas (`"johnnydepp"`, `"johnnygalecki"`). De este modo, cada entidad se convierte en un token único.

In [8]:
# Función para normalizar texto: convertir a minúsculas y quitar espacios
def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        if isinstance(x, str):
            return str.lower(x.replace(" ", ""))
        else:
            return ''

features = ['cast', 'keywords', 'director', 'genres']
for feature in features:
    df[feature] = df[feature].apply(clean_data)

## 6. Creación de la "Metadata Soup"
Unimos los actores, el director, las palabras clave y los géneros en una sola cadena de texto por cada película.

In [9]:
# Crear la sopa de metadatos
def create_soup(x):
    return ' '.join(x['keywords']) + ' ' + ' '.join(x['cast']) + ' ' + x['director'] + ' ' + ' '.join(x['genres'])

df['soup'] = df.apply(create_soup, axis=1)
df[['title', 'soup']].head(2)

,title,soup
0,Toy Story,jealousy toy boy tomhanks timallen donrickles ...
1,Jumanji,boardgame disappearance basedonchildren'sbook ...


## 7. Vectorización con CountVectorizer y Similitud Coseno

A diferencia del recomendador basado en sinopsis, aquí utilizamos **`CountVectorizer`** en lugar de `TF-IDF`.
* **Razón:** No queremos penalizar ni reducir el peso de directores o actores prolíficos (como Christopher Nolan o Tom Hanks) solo porque aparecen en muchas películas. La repetición o presencia de un nombre debe mantener su valor completo.

In [10]:
# Vectorizar con CountVectorizer
count = CountVectorizer(stop_words='english')
count_matrix = count.fit_transform(df['soup'])

print("Dimensiones de la matriz de conteo:", count_matrix.shape)

Dimensiones de la matriz de conteo: (46626, 73880)


In [11]:
# Calcular matriz de similitud coseno
cosine_sim2 = cosine_similarity(count_matrix, count_matrix)
print("Dimensiones de la matriz de similitud cosine_sim2:", cosine_sim2.shape)

# Resetear el índice del DataFrame y crear el mapeo inverso actualizado
df = df.reset_index()
indices = pd.Series(df.index, index=df['title'])

Dimensiones de la matriz de similitud cosine_sim2: (46626, 46626)


## 8. Recomendaciones con el Modelo Mejorado

In [12]:
# Reutilizar la función get_recommendations pasando cosine_sim2
def get_recommendations(title, cosine_sim=cosine_sim2, df=df, indices=indices):
    if title not in indices:
        return f"Película '{title}' no encontrada en el catálogo."
        
    idx = indices[title]
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11]
    movie_indices = [i[0] for i in sim_scores]
    return df['title'].iloc[movie_indices]

In [13]:
# Prueba 1: The Dark Knight Rises
get_recommendations('The Dark Knight Rises', cosine_sim2)

12541      The Dark Knight
10170        Batman Begins
9271                Shiner
9834       Amongst Friends
7732              Mitchell
516      Romeo Is Bleeding
11411         The Prestige
24040            Quicksand
24984             Deadfall
41041                 Sara
Name: title, dtype: str

In [14]:
# Prueba 2: The Godfather
get_recommendations('The Godfather', cosine_sim2)

1926            The Godfather: Part III
1187             The Godfather: Part II
15534                   The Rain People
18866                         Last Exit
34458                              Rege
35772            Manuscripts Don't Burn
35773            Manuscripts Don't Burn
7961     The Night of the Following Day
18187                 The Son of No One
28637            In the Name of the Law
Name: title, dtype: str

## Conclusiones 
* **Mejora notable:** Al buscar *The Dark Knight Rises*, ahora encontramos no solo películas de Batman (*The Dark Knight*, *Batman Begins*), sino también otras películas dirigidas por **Christopher Nolan** (*The Prestige*) u obras de tono y elenco afín.
* Para *The Godfather*, ahora se recomiendan de forma prioritaria *The Godfather: Part II*, *The Godfather: Part III* y otras películas dirigidas por Francis Ford Coppola o protagonizadas por Al Pacino / Marlon Brando.

